In [ ]:
# Base PINN:
# Copy this file's contents into your own Jupyter/Colab notebook and adapt it for your own model
# What to keep when you copy this:
# BasePINN: the encode_input()/trunk_forward() hook design.
# The shared hyperparameter block below.

# What to change for your variant:
# Write your own class (like VanillaPINN) that fills in encode_input() and trunk_forward() with your architecture
# forward() stays the same.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Hyperparameters
# Shared across every architecture. Only variant-specific parameters should be added separately in your own copy unless the whole team agrees
MU = 1.0 # nonlinearity factor
T_FINAL = 20.0 # the edge of the interval for the oscillator simulation
X0 = 1.0 # initial position x(0)
V0 = 0.0 # initial velocity x'(0)
WINDOW_SIZE = 2.0 # length of each time-marching window
LAYERS = (1, 32, 32, 32, 32, 1) # shared network depth/width

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class BasePINN(nn.Module):
    """
    Shared skeleton for every PINN architecture
    Subclasses only override two functions:
    - encode_input(t): raw time t -> whatever representation this architecture's trunk expects.
    - trunk_forward(encoded): the layer stack mapping the encoded input to a raw scalar output.
    """
    def __init__(self):
        super().__init__()

    def encode_input(self, t):
        raise NotImplementedError(
            "Subclasses of BasePINN must implement encode_input()."
        )

    def trunk_forward(self, encoded):
        raise NotImplementedError(
            "Subclasses of BasePINN must implement trunk_forward()."
        )

    def forward(self, t):
        encoded = self.encode_input(t)
        return self.trunk_forward(encoded)

    @staticmethod
    def init_linear(linear):
        nn.init.xavier_normal_(linear.weight)
        nn.init.zeros_(linear.bias)

In [ ]:
class VanillaPINN(BasePINN):
    """
    The standard PINN architecture: a plain tanh MLP.
    Reference implementation of the BasePINN pattern.
    Copy this class's shape when writing SIREN or Fourier-feature variants.
    """
    def __init__(self, t_start, window_size, layers=LAYERS):
        super().__init__()
        self.t_start = t_start
        self.window_size = window_size

        self.trunk = nn.ModuleList()
        for i in range(len(layers) - 1):
            linear = nn.Linear(layers[i], layers[i + 1])
            self.init_linear(linear)
            self.trunk.append(linear)
        self.activation = torch.tanh

    def encode_input(self, t):
        tau = t - self.t_start
        return (2.0 * tau / self.window_size) - 1.0

    def trunk_forward(self, encoded):
        out = encoded
        for layer in self.trunk[:-1]:
            out = self.activation(layer(out))
        return self.trunk[-1](out)